# ELO Model

The ELO system is based on the relative strength between two teams. This strength is measured on a continuous scale and the expected results for a team is defined by a logistic function for the qualification strength difference

$$W_e = \frac{1}{10^{-d_r / 400} + 1}$$

where $d_r = R_{\text{local}} - R_{\text{visitor}} + d_{\text{home}}$, $d_{\text{home}}$ is a local advantage factor and $R$ the rescptive ELO rank value.

After finishing a match, the ELO strength is updated:

$$R_{\text{new}} = R_{\text{ond}} + K \times (W - W_e)$$

where $W$ is the result ($1$ for local win, $0.5$ for a draw and $0$ for visitor win). The $K$ parameter is used to control the maximum impact for just one match. And symmetrically, the visitor lost the points won for the local.

For their nature, the strength parameters are fitted sequentially and chronologically. The initial values are usually fixed at 1500 points.   

The K factor can be weighted by the difference in the result, showing a bigger impact for a bigger score difference
$$
K_{modified} = K_{base} \times M(n)$$

where $M(n)$ is a function empirically calibrated
$$
M(n)= = \begin{cases} 
1 & \text{if } N\leq 1 \\ 
1.5 & \text{if } N=2 \\ 
1.75 & \text{if } N=3 \\ 
1.75+\frac{N-3}{8} & \text{if } N\geq 4 \end{cases}
$$


## Results simulation


For obtaining the probability of each outcome (W-D-L):
$$P(\text{Local victory}) = \frac{1}{1 + 10^{-(d_R - c) / 400}}\   \   \    P(\text{Visitor victory}) = \frac{1}{1 + 10^{(d_R + c) / 400}}$$

where $c$ is a margin for a draw and then
$$P(\text{Draw}) = 1 - P(\text{Local victory}) - P(\text{Visitor victory})$$

In particular, because in a World Cup, the home effect is negligible, we will fix $d_{\text{home}=0$


## Evaluating the prediction

For prediction evaluation we use the Ranked Probability Score (RPS)
$$RPS = \frac{1}{r-1} \sum_{i=1}^{r-1} (P_i - O_i)^2$$
where:
- $P_i$: is the cumulative sum of predicted probabilities till category $i$.
- $O_i$: is the cumulative sum of the actual results till category $i$.
- $r$ is the total of possible results, in our case 3 (home win, draw, away win).
This metric is calculated for each predicted match.

## Training and validating model

In order to obtain the optimal values  of $k_{base}$, $d_{\text{home}}$ and $c$, we should minimize the RPS. For a robust result, we implement a burn-in period:
Phase 1: Burn-in: the first $N$ matches are used to obtain a brief approximation of the relative strength of the teams. Update the ratings without considering the RPS.
Phase 2: Scoring: after passing the first $N$ matches, we start to compute the RPS. This allows for optimizing parameters over quite reliable ranks for the teams.

This is important because, without the burn-in, the optimizer would pick a $K$ value that is artificially high just to "fix" the initial 1500 bias, leading to overfitting and poor predictive power in the validation set.

We consider the train set the matches prior to the FIFA World Cup 2022 and the burn-in period till the FIFA World Cup 2018.

Finally, for the World Cup matches, we set $d_{\text{home}}$ to $0$ because most teams do not have home-field advantage.

To validate the model, we compute the RPS for the matches in the FIFA World Cup 2022.



In [14]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. PREPARACIÓN ACELERADA DE DATOS
df = pd.read_parquet('../data/processed/matches.parquet')
df = df.sort_values('date').reset_index(drop=True)

# Creamos un mapeo de equipos a IDs enteros para máxima velocidad
all_teams = sorted(list(set(df['home_team']).union(set(df['away_team']))))
team_to_id = {team: i for i, team in enumerate(all_teams)}
id_to_team = {i: team for team, i in team_to_id.items()}

df['h_id'] = df['home_team'].map(team_to_id)
df['a_id'] = df['away_team'].map(team_to_id)

# --- FUNCIONES NÚCLEO (Optimizadas para cálculos matemáticos) ---

def get_k_multiplier(n):
    n = abs(n)
    if n <= 1: return 1.0
    if n == 2: return 1.5
    if n == 3: return 1.75
    return 1.75 + (n - 3) / 8.0

# --- FASE 1: ENTRENAMIENTO (ULTRA RÁPIDO) ---
def train_parameters(df_train, num_teams, burn_in=1000):
    # Convertimos el dataframe a un array de NumPy una sola vez
    # Columnas: 0:h_id, 1:a_id, 2:h_score, 3:a_score
    data_array = df_train[['h_id', 'a_id', 'home_score', 'away_score']].values.astype(int)
    
    def objective(params):
        k_base, hfa, draw_margin = params
        # Usamos un array de NumPy para los ratings (acceso instantáneo por índice)
        ratings = np.full(num_teams, 1500.0)
        total_rps = 0.0
        count = 0
        
        for i in range(len(data_array)):
            h_id, a_id, h_s, a_s = data_array[i]
            
            # Cálculo de Elo
            delta = ratings[h_id] + hfa - ratings[a_id]
            
            # Predicción
            p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
            p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
            p_draw = max(0.0, 1.0 - p_home - p_away)
            
            # RPS
            actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
            e = 1.0 if actual == 0 else 0.5 if actual == 1 else 0.0
            
            # Cálculo de RPS manual (más rápido que llamar a función)
            p_cum_0 = p_home
            p_cum_1 = p_home + p_draw
            e_cum_0 = 1.0 if actual == 0 else 0.0
            e_cum_1 = 1.0 if actual <= 1 else 0.0
            if i >= burn_in:
                total_rps += ((p_cum_0 - e_cum_0)**2 + (p_cum_1 - e_cum_1)**2) / 2.0
                count += 1
            
            # Actualización
            m_n = get_k_multiplier(h_s - a_s)
            e_home = 1 / (1 + 10 ** (-delta / 400))
            k_mod = k_base * m_n
            
            shift = k_mod * (e - e_home)
            ratings[h_id] += shift
            ratings[a_id] -= shift
            
        return total_rps / count if count > 0 else 0

    print("Iniciando optimización pesada...")
    res = minimize(objective, [20.0, 40.0, 90.0], 
                   bounds=[(5, 80), (0, 150), (10, 200)], method='L-BFGS-B')
    return res.x

# --- FASE 2: VALIDACIÓN (Mantiene tu visualización) ---
def run_static_validation(df_all, train_params, start_eval_idx, end_eval_idx, large_visualization=True):
    k_base, hfa, draw_margin = train_params
    num_teams = len(all_teams)
    ratings = np.full(num_teams, 1500.0)
    rps_scores = []
    
    # Pre-extraemos valores para evitar iterrows() incluso en validación
    # pero mantenemos acceso a la fila para la visualización
    print(f"Validación: K={k_base:.1f}, HFA={hfa:.1f}, Margin={draw_margin:.1f}")
    print("-" * 85)
    df_window = df_all.iloc[:end_eval_idx]

    for i, row in df_window.iterrows():
        h_id, a_id = row['h_id'], row['a_id']
        h_s, a_s = int(row['home_score']), int(row['away_score'])
        
        # 1. Predicción
        delta = ratings[h_id] + hfa - ratings[a_id]
        p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
        p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
        p_draw = max(0.0, 1.0 - p_home - p_away)
        probs = [p_home, p_draw, p_away]
        
        actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
        r_h_prev, r_a_prev = ratings[h_id], ratings[a_id]

        if i >= start_eval_idx:
            # Calcular RPS
            p_cum = np.cumsum(probs)
            e_cum = [1.0 if actual == 0 else 0.0, 1.0 if actual <= 1 else 0.0]
            current_rps = ((p_cum[0]-e_cum[0])**2 + (p_cum[1]-e_cum[1])**2)/2.0
            rps_scores.append(current_rps)
            
            if large_visualization:
                if (row['home_team'] == 'Argentina') or (row['away_team'] == 'Argentina'):
                    status = "✅" if np.argmax(probs) == actual else "❌"
                    print(f"[{row['date'].strftime('%Y-%m-%d')}] {row['home_team']:<20} {h_s}-{a_s:^3} {row['away_team']:>20} | {status}")
                    print(f"  └─ Pred: L: {p_home:>6.1%} | E: {p_draw:>6.1%} | V: {p_away:>6.1%}")
                    print(f"  └─ Elo: {r_h_prev:.0f} vs {r_a_prev:.0f} | RPS: {current_rps:.4f}")
                    print("-" * 85)
            elif i % 5000 == 0:
                print(f"Procesando: {i}/{len(df_all)}...")

        # 2. Actualizar
        outcome = 1.0 if h_s > a_s else 0.5 if h_s == a_s else 0.0
        e_home = 1 / (1 + 10 ** (-delta / 400))
        k_mod = k_base * get_k_multiplier(h_s - a_s)
        
        shift = k_mod * (outcome - e_home)
        ratings[h_id] += shift
        ratings[a_id] -= shift

    return np.mean(rps_scores)

# ==========================================
# EJECUCIÓN
# ==========================================
if __name__ == '__main__':
    # Define aquí tu ventana de laboratorio
    WARMING_ELO = 18948
    TRAIN_END = 21729
    VALIDATION_END = 21793
    
    df_train = df.iloc[:TRAIN_END]
    
    
    # 1. Calibramos las "reglas" del modelo
    best_params = train_parameters(df.iloc[:TRAIN_END], len(all_teams),WARMING_ELO)    

    best_params[1] = 0
    
    # 2. Validamos el rendimiento en el resto del tiempo
    final_rps = run_static_validation(df,
                                      best_params,
                                      start_eval_idx=TRAIN_END,
                                      end_eval_idx=VALIDATION_END,
                                      large_visualization=True
                                      )
    
    print(f"\nRPS Promedio Global: {final_rps:.4f}")    

Iniciando optimización pesada...
Validación: K=32.1, HFA=0.0, Margin=108.7
-------------------------------------------------------------------------------------
[2022-11-22] Argentina            1- 2          Saudi Arabia | ❌
  └─ Pred: L:  57.8% | E:  24.9% | V:  17.3%
  └─ Elo: 2011 vs 1848 | RPS: 0.5088
-------------------------------------------------------------------------------------
[2022-11-26] Argentina            2- 0                Mexico | ✅
  └─ Pred: L:  41.9% | E:  29.7% | V:  28.4%
  └─ Elo: 1988 vs 1936 | RPS: 0.2088
-------------------------------------------------------------------------------------
[2022-11-30] Poland               0- 2             Argentina | ✅
  └─ Pred: L:  13.3% | E:  21.6% | V:  65.0%
  └─ Elo: 1792 vs 2008 | RPS: 0.0700
-------------------------------------------------------------------------------------
[2022-12-03] Argentina            2- 1             Australia | ✅
  └─ Pred: L:  58.3% | E:  24.7% | V:  17.0%
  └─ Elo: 2019 vs 1852 | RPS: 

In [ ]:
df_2022.head(64)